# Benchmark Trip Helper — trajets calculés par minute

Mesure le débit brut du moteur de routage **indépendamment du pipeline LLM**.

| Backend | Endpoint par défaut | Ce qu'on mesure |
|---------|--------------------|-----------------|
| **OSMnx** | `POST http://localhost:8090/route` | Routage direct pied/vélo/voiture |
| **OTP** | `POST http://localhost:8080/otp/transmodel/v3` | Routage TC (GraphQL) |

**Protocole** : pour chaque palier de concurrence, on tire `N_PAIRS` paires O/D aléatoires
depuis la population et on les envoie en rafale sur `DURATION_S` secondes. On mesure :
- Trajets calculés / minute
- Latence p50 / p95
- Taux d'erreur

> **Prérequis** : le service OSMnx (port 8090) et/ou OTP (port 8080) doivent être démarrés.

In [ ]:
import asyncio
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import aiohttp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Backend à tester : "osmnx" ou "otp"
BACKEND          = "osmnx"

# Endpoints
OSMNX_URL        = "http://localhost:8090/route"
OTP_URL          = "http://localhost:8080/otp/transmodel/v3"

# Population source (paires O/D)
POPULATION_FILE  = Path("../../data/eqasim_output/toulouse_population_1000.json")

# Mode OSMnx : "foot" | "bicycle" | "car"
OSMNX_MODE       = "foot"

# Timestamp de départ simulé (cohérent avec la population Toulouse)
SIM_TIMESTAMP    = 1_775_800_000

# Durée de chaque palier (secondes)
DURATION_S       = 60

# Paliers de concurrence à tester
CONCURRENCY_STEPS = [1, 2, 4, 8, 16, 32, 64]

# Timeout par requête (secondes)
REQUEST_TIMEOUT_S = 30

# Graine aléatoire pour reproductibilité
RANDOM_SEED      = 42

# Seuil p95 au-delà duquel on arrête (secondes)
STOP_P95_S       = 20.0

In [ ]:
# ── Chargement de la population ────────────────────────────────────────────────
raw = json.loads(POPULATION_FILE.read_text())

locations = []
for p in raw:
    home = p["identity"]["home"]
    locations.append({"lat": home["lat"], "lon": home["lon"]})

rng = random.Random(RANDOM_SEED)
rng.shuffle(locations)

# Pré-générer les paires O/D (on en a besoin en grande quantité)
def sample_pair(rng):
    origin, dest = rng.sample(locations, 2)
    return origin, dest

print(f"Population chargée : {len(locations)} localisations")
o, d = sample_pair(rng)
print(f"Exemple paire O/D : {o} → {d}")

In [ ]:
# ── Construction des payloads selon le backend ─────────────────────────────────
CONGESTION_DT = datetime.fromtimestamp(SIM_TIMESTAMP, tz=timezone.utc).isoformat()

# Requête GraphQL OTP minimale (transit uniquement, mode foot access/egress)
_OTP_QUERY = """
query trip($from: Location!, $to: Location!, $dateTime: DateTime, $numTripPatterns: Int, $searchWindow: Int) {
  trip(
    from: $from
    to: $to
    dateTime: $dateTime
    numTripPatterns: $numTripPatterns
    searchWindow: $searchWindow
    modes: {
      accessMode: foot
      egressMode: foot
      transportModes: [
        {transportMode: bus}
        {transportMode: metro}
        {transportMode: tram}
      ]
    }
  ) {
    tripPatterns {
      duration
      startTime
      endTime
    }
  }
}
"""

def make_osmnx_payload(origin: dict, destination: dict) -> dict:
    return {
        "origin":      {"lat": origin["lat"],      "lon": origin["lon"]},
        "destination": {"lat": destination["lat"], "lon": destination["lon"]},
        "mode":        OSMNX_MODE,
        "congestion_dt": CONGESTION_DT,
    }

def make_otp_payload(origin: dict, destination: dict) -> dict:
    dt = datetime.fromtimestamp(SIM_TIMESTAMP, tz=timezone.utc).isoformat()
    return {
        "query": _OTP_QUERY,
        "operationName": "trip",
        "variables": {
            "from": {"coordinates": {"latitude": origin["lat"], "longitude": origin["lon"]}},
            "to":   {"coordinates": {"latitude": destination["lat"], "longitude": destination["lon"]}},
            "dateTime": dt,
            "numTripPatterns": 5,
            "searchWindow": 30,
        },
    }

make_payload = make_osmnx_payload if BACKEND == "osmnx" else make_otp_payload
endpoint_url = OSMNX_URL if BACKEND == "osmnx" else OTP_URL
print(f"Backend : {BACKEND.upper()}  →  {endpoint_url}")
print(f"Payload exemple : {json.dumps(make_payload(locations[0], locations[1]), indent=2)[:300]}...")

In [ ]:
# ── Fonctions de mesure ────────────────────────────────────────────────────────

async def send_one(session: aiohttp.ClientSession, origin: dict, dest: dict) -> dict:
    """Envoie une requête et retourne {latency, error}."""
    payload = make_payload(origin, dest)
    t0 = time.perf_counter()
    try:
        async with session.post(
            endpoint_url,
            json=payload,
            timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_S),
        ) as resp:
            body = await resp.json(content_type=None)
            latency = time.perf_counter() - t0
            if resp.status >= 400:
                return {"latency": latency, "error": f"http_{resp.status}"}
            # OTP renvoie des erreurs dans le corps JSON
            if BACKEND == "otp" and isinstance(body, dict) and body.get("errors"):
                return {"latency": latency, "error": str(body["errors"][0].get("message", "otp_error"))[:80]}
            return {"latency": latency, "error": None}
    except asyncio.TimeoutError:
        return {"latency": REQUEST_TIMEOUT_S, "error": "timeout"}
    except Exception as e:
        return {"latency": time.perf_counter() - t0, "error": str(e)[:80]}


async def run_palier(
    session: aiohttp.ClientSession,
    concurrency: int,
    duration_s: float,
    rng_state: random.Random,
) -> list[dict]:
    """
    Maintient `concurrency` requêtes en vol simultané pendant `duration_s` secondes.
    Dès qu'une requête se termine, on en lance immédiatement une nouvelle (closed-loop).
    """
    results = []
    semaphore = asyncio.Semaphore(concurrency)
    deadline = time.perf_counter() + duration_s

    async def worker():
        while time.perf_counter() < deadline:
            origin, dest = rng_state.sample(locations, 2)
            async with semaphore:
                r = await send_one(session, origin, dest)
                results.append(r)

    workers = [asyncio.create_task(worker()) for _ in range(concurrency)]
    await asyncio.gather(*workers, return_exceptions=True)
    return results


def compute_metrics(results: list[dict], duration_s: float) -> dict:
    if not results:
        return {}
    latencies = np.array([r["latency"] for r in results])
    errors    = [r for r in results if r["error"] is not None]
    n         = len(results)
    n_err     = len(errors)
    error_types = {}
    for r in errors:
        k = r["error"]
        error_types[k] = error_types.get(k, 0) + 1
    return {
        "n_requests":    n,
        "trips_per_min": round(n / duration_s * 60, 1),
        "p50_s":         float(np.percentile(latencies, 50)),
        "p95_s":         float(np.percentile(latencies, 95)),
        "p99_s":         float(np.percentile(latencies, 99)),
        "mean_s":        float(np.mean(latencies)),
        "error_rate":    n_err / n,
        "error_types":   error_types,
    }

In [ ]:
# ── Vérification de la connectivité ───────────────────────────────────────────
async def health_check():
    async with aiohttp.ClientSession() as session:
        if BACKEND == "osmnx":
            try:
                async with session.get(
                    OSMNX_URL.replace("/route", "/health"),
                    timeout=aiohttp.ClientTimeout(total=5)
                ) as r:
                    print(f"OSMnx health : HTTP {r.status}")
            except Exception as e:
                print(f"OSMnx health : ERREUR {e}")

        # Test d'un trajet réel
        o, d = locations[0], locations[1]
        r = await send_one(session, o, d)
        status = "OK" if r["error"] is None else f"ERREUR {r['error']}"
        print(f"Test trajet   : {status}  (latence {r['latency']*1000:.0f} ms)")

await health_check()

In [ ]:
# ── Benchmark principal ────────────────────────────────────────────────────────
records = []
rng_run = random.Random(RANDOM_SEED)

connector = aiohttp.TCPConnector(limit=0)

async def run_benchmark():
    async with aiohttp.ClientSession(connector=connector) as session:
        for concurrency in CONCURRENCY_STEPS:
            print(f"\nPalier concurrence={concurrency} — tir {DURATION_S}s...")
            results = await run_palier(session, concurrency, DURATION_S, rng_run)
            m = compute_metrics(results, DURATION_S)
            m["concurrency"] = concurrency
            records.append(m)
            print(
                f"  n={m['n_requests']:>5}  "
                f"trips/min={m['trips_per_min']:>7.1f}  "
                f"p50={m['p50_s']*1000:>6.0f}ms  "
                f"p95={m['p95_s']*1000:>6.0f}ms  "
                f"err={m['error_rate']*100:>5.1f}%"
            )
            if m["p95_s"] >= STOP_P95_S:
                print(f"  ⚠  ARRÊT : p95={m['p95_s']:.1f}s ≥ seuil {STOP_P95_S}s")
                break

await run_benchmark()

In [ ]:
# ── Tableau des résultats ─────────────────────────────────────────────────────
df = pd.DataFrame(records)
df["error_pct"] = df["error_rate"] * 100
df[["concurrency", "n_requests", "trips_per_min", "mean_s", "p50_s", "p95_s", "error_pct"]].round(2)

In [ ]:
# ── Visualisation 1 : débit (trajets/min) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["concurrency"], df["trips_per_min"],
        marker="o", markersize=6, color="steelblue", linewidth=2, label="trajets/min")

for _, row in df.iterrows():
    ax.annotate(
        f"{row['trips_per_min']:.0f}",
        (row["concurrency"], row["trips_per_min"]),
        textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8
    )

ax.set_xlabel("Concurrence (requêtes simultanées)")
ax.set_ylabel("Trajets calculés / minute")
ax.set_title(f"Débit du trip helper — backend {BACKEND.upper()} / mode {OSMNX_MODE if BACKEND == 'osmnx' else 'TC'}")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"trip_benchmark_{BACKEND}_{OSMNX_MODE if BACKEND == 'osmnx' else 'tc'}_throughput.png", dpi=150)
plt.show()

In [ ]:
# ── Visualisation 2 : latences p50/p95 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["concurrency"], df["p50_s"] * 1000,
        marker="o", markersize=5, color="steelblue", label="p50")
ax.plot(df["concurrency"], df["p95_s"] * 1000,
        marker="s", markersize=5, color="darkorange", label="p95")
ax.axhline(STOP_P95_S * 1000, color="red", linestyle="--", linewidth=1.2, label=f"seuil {STOP_P95_S}s")

ax.set_xlabel("Concurrence (requêtes simultanées)")
ax.set_ylabel("Latence (ms)")
ax.set_title(f"Latences — backend {BACKEND.upper()} / mode {OSMNX_MODE if BACKEND == 'osmnx' else 'TC'}")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"trip_benchmark_{BACKEND}_{OSMNX_MODE if BACKEND == 'osmnx' else 'tc'}_latency.png", dpi=150)
plt.show()

In [ ]:
# ── Synthèse ──────────────────────────────────────────────────────────────────
best = df.loc[df["error_rate"] < 0.05, "trips_per_min"].idxmax() if not df.empty else None
print("=" * 60)
print(f"BENCHMARK TRIP HELPER — {BACKEND.upper()} / {OSMNX_MODE if BACKEND == 'osmnx' else 'TC'}")
print(f"Durée par palier : {DURATION_S}s  |  Seuil p95 arrêt : {STOP_P95_S}s")
print("=" * 60)
if best is not None:
    r = df.loc[best]
    print(f"  Débit max (err<5%) : {r['trips_per_min']:.0f} trajets/min  "
          f"à concurrence={r['concurrency']:.0f}")
    print(f"  Latence à ce palier : p50={r['p50_s']*1000:.0f}ms  p95={r['p95_s']*1000:.0f}ms")
else:
    print("  Aucun palier sous le seuil d'erreur.")
print()
print(df[["concurrency", "trips_per_min", "p50_s", "p95_s", "error_pct"]].round(2).to_string(index=False))

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────────────
csv_path = f"trip_benchmark_{BACKEND}_{OSMNX_MODE if BACKEND == 'osmnx' else 'tc'}_results.csv"
df.drop(columns=["error_types"], errors="ignore").to_csv(csv_path, index=False)
print(f"Résultats sauvegardés : {csv_path}")